In [ ]:
import sys, os
sys.path.append("..")

from app.chains.chat_pipeline_mcp import chat_once
s = await chat_once("What is postpartum care ?", "696ba3111cf1f3260172e177")
s

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ChatResponse(request_id='371417c5', session_id='7eadc56f-fa52-449c-b95d-d97a51ad8807', answer="Harsha, I can sense that you might be feeling a bit overwhelmed with your postpartum recovery and wondering about unrelated things, like Rahul Dravid. It's completely normal to have your mind wander, especially when you're taking care of a new baby and managing your physical and emotional well-being. Rahul Dravid is a former Indian cricketer, but I understand that's not directly related to your current concerns.\n\nLet's focus on you for now. I see that your recovery trend is improving, which is great news! However, I also notice that you're still in the YELLOW zone for physical recovery and lactation, and the RED zone for emotional well-being. Remember, you have a strong support system with your family, and that's a big plus.\n\nTake small steps each day to prioritize your self-care, Harsha. Make sure to rest when you can, and don't hesitate to ask for help with the baby or household chores.

In [ ]:
"""
Test script for recommendations_tool

Tests fetching the last 3 recommendations and formatting for the chatbot.

Run: python3 tests/test_recommendations.py
"""

import sys
import os

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(".env"))))

from dotenv import load_dotenv
load_dotenv()

from app.mcp.tools.get_active_recommendations_tool import (
    get_active_recommendations,
    format_recommendations_for_prompt,
    print_recommendations_summary
)
from app.mcp.db_connection import get_recommendation_history_collection

from app.mcp.db_connection import get_users_collection


def test_user_profile_tool():
    """Test the user profile fetching functionality"""
        

def test_recommendations_tool():
    """Test the recommendations fetching functionality"""
    print("\n" + "="*60)
    print("Testing Recommendations Tool")
    print("="*60 + "\n")
    print("\n" + "="*60)
    print("Testing User Profile Tool")
    print("="*60 + "\n")
        # First, let's find a real user in the database to test with
    users = get_users_collection()
    print(users, "223")
    sample_user = users.find_one({})
    print(sample_user, "user sample")
    try:
        
        # Find a user who has recommendation history
        rec_history = get_recommendation_history_collection()
        print(rec_history, "2232")
        sample_rec = rec_history.find_one({})
        print(sample_rec, "3232")
        
        if not sample_rec:
            print("ℹ️  No recommendation history in database yet.")
            print("   Users need to complete weekly check-ins first.")
            return False
        
        # Get the userId from the sample
        user_id = sample_rec.get("userId")
        if not user_id:
            print("❌ Sample recommendation has no userId field")
            return False
        
        user_id_str = str(user_id)
        
        print(f"✅ Found user with recommendation history")
        print(f"   User ID: {user_id_str}")
        print()
        
        # Test 1: Fetch recommendations
        print("-" * 60)
        print("Test 1: Fetching Last 3 Recommendations")
        print("-" * 60 + "\n")
        
        recs_data = get_active_recommendations(user_id_str, limit=3)
        
        if recs_data.get("found"):
            print(f"✅ Successfully fetched recommendations!")
            print(f"   Count: {recs_data['count']}")
            print(f"   Trend: {recs_data['trend']}")
            print()
            
            # Show each recommendation briefly
            for i, rec in enumerate(recs_data["recommendations"]):
                marker = "← CURRENT" if i == len(recs_data["recommendations"]) - 1 else ""
                print(f"   Week {rec['week']}: Score {rec['finalScore']}, {rec['zone']} zone {marker}")
        else:
            print(f"❌ Failed to fetch: {recs_data.get('error')}")
            return False
        
        # Test 2: Format for LLM
        print("\n" + "-" * 60)
        print("Test 2: Formatting for LLM Prompt")
        print("-" * 60 + "\n")
        
        formatted = format_recommendations_for_prompt(recs_data)
        print("✅ Formatted context:")
        print()
        print(formatted)
        print()
        
        # Test 3: Verify latest recommendation has content
        print("-" * 60)
        print("Test 3: Checking Recommendation Content")
        print("-" * 60 + "\n")
        
        latest = recs_data.get("latest")
        if latest and latest.get("content"):
            content = latest["content"]
            print("✅ Latest recommendation has content:")
            print(f"   Title: {content.get('title', 'N/A')[:60]}...")
            print(f"   Going Well: {content.get('goingWell', 'N/A')[:60]}...")
            print(f"   Focus: {content.get('needsHelp', 'N/A')[:60]}...")
        else:
            print("⚠️  Latest recommendation missing content")
            print("   (This might be normal if recommendations haven't been fully set up)")
        
        # Test 4: Pretty print
        print("\n" + "-" * 60)
        print("Test 4: Pretty Print Summary")
        print("-" * 60)
        
        print_recommendations_summary(user_id_str)
        
        return True
        
    except Exception as e:
        print(f"❌ Error during testing: {str(e)}")
        import traceback
        traceback.print_exc()
        return False


def test_with_specific_user():
    """Test with a specific user ID"""
    print("\n" + "="*60)
    print("Test with Specific User ID")
    print("="*60 + "\n")
    
    user_id = input("Enter a user_id to test (or press Enter to skip): ").strip()
    
    if not user_id:
        print("Skipped.")
        return
    
    print()
    print_recommendations_summary(user_id)


def main():
    """Run all tests"""
    print("\n🏥 Viva Mama - Recommendations Tool Tests")
    print("="*60)
    
    success = test_recommendations_tool()
    
    print("\n" + "="*60)
    if success:
        print("🎉 All tests passed!")
        print("="*60 + "\n")
        print("The recommendations tool is working correctly.")
        print()
        print("What this means:")
        print("✅ Can fetch last 3 recommendations")
        print("✅ Can calculate recovery trend (improving/stable/declining)")
        print("✅ Can link to full recommendation content")
        print("✅ Can format for chatbot consumption")
        print()
        print("This allows the chatbot to:")
        print("  - Reference current week's focus")
        print("  - Acknowledge progress from previous weeks")
        print("  - Celebrate improvements")
        print("  - Address ongoing concerns")
        print()
        
        test_with_specific_user()
    else:
        print("⚠️  Some tests failed")
        print("="*60 + "\n")
    
    return 0 if success else 1


if __name__ == "__main__":
    exit(main())